# Lab 2 — The Trading League
### EP0418 · Topic 2 · 2 practical hours (Week 2) · the league stays open until the end of Week 7 · results at Lab 7

**Scoreboard:** https://stanleysp-eng.github.io/EP0418-labs/

**You need:** `ep0418.py`, which the notebook fetches, and an internet connection
for the prices. The Lab 2 sheet is the player's guide: the plan for the week, ideas
to start from, the three traps, and the write-up.

---
This lab opens a six-week competition with one rule that matters: **you are
scored on prices nobody has seen.**

The market is **SPY**, the fund that tracks the 500 largest US companies and the
most traded security in the world. The notebook fetches its last five years of
daily prices into your own Colab. You have until the end of Week 7 (the exact
closing time is on the scoreboard) to build a rule on them, with any AI tool,
method or library you like, and you can replace your entry as often as you like
until then; each lab hands you a new tool for it. Nothing is graded. Then the
market does what it does for four weeks, and every entry is run on SPY's real
prices from those weeks, prices that did not exist when you entered. The
rule that looks best on the past is usually not the one that wins. Finding out
why is the point.

**The rules**

1. Your entry is a function `strategy(prices)` that returns a position for every
   trading day: 1 (hold SPY), 0 (cash) or -1 (short).
2. A decision for a day may use that day's prices and earlier. Never later.
3. The scorer is locked. It delays your position by one day, charges 0.1% of the
   amount traded each time you change position, and reports risk-adjusted return
   (the Sharpe ratio) after costs. Read it in Task 3; do not edit it.
4. Buy-and-hold is an entrant. Beating it after costs on the scored week is the prize.
5. Your strategy must change position at least four times over the five years.
   Buy-and-hold under a new name is not an entry.
6. The hidden four weeks are ranked on return after costs; the five-year board
   shows the Sharpe ratio and the return.
7. Two podiums: best hidden period, and best write-up (Task 6).
8. Enter as often as you like before the deadline; the latest entry counts. The
   scoreboard shows every entry's five-year score the moment it is submitted. The
   hidden board appears after the four weeks have been scored.

Paper trading only. Prices come from Yahoo Finance through **yfinance**, a free,
unofficial library for personal, educational use. Nothing here is investment
advice, and a live broker earns nothing.

## Putting the assistant's code into a TODO cell

A TODO cell has a line ending in `...`. Replace **only that line** with the
assistant's code, and keep the name on the left (`s`, `clean`, `H`). Delete any
`import` or `read_csv` lines it adds; the data is already loaded. If it used a
different name for the result, add one line at the end to rename it, e.g.
`s = hourly_df`.

---
## How every lab in this module works

You will **not** write this code from scratch. Ask your AI assistant to write it.
Your job is the part it cannot do: deciding what the right answer looks like,
reading what it gave you, and proving it did what you asked.

Every prompt in these notebooks opens with the rules that matter for that task:
do not drop rows, show the row count before and after, treat -999 as missing,
explain the code in plain words. Keep those lines when you adapt a prompt, and
start your own prompts the same way.

At the foot of each task is a **verification contract** — a few `check.*` lines.
**You may not edit them.** The AI can write any implementation it likes; it cannot
satisfy these by accident.

If a check fails, read the message. It tells you what was found and what to try
next. Failing checks are the normal state of a lab in progress, not a sign you
are behind.

> **First: File → Save a copy in Drive, and work in your copy.** This page opened from GitHub, and changes here are not saved. Reopen your copy from Google Drive next time.

In [ ]:
import os, urllib.request
REPO = "https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/"


def fetch(*names):
    """Download any course file that is not beside the notebook yet."""
    for name in names:
        if os.path.exists(name):
            continue
        try:
            urllib.request.urlretrieve(REPO + name, name)
            print("fetched", name)
        except Exception:
            print(f"Could not download {name}. Upload it with the folder icon on the left.")
            from google.colab import files
            files.upload()


fetch("ep0418.py")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from ep0418 import check, show


def filled(*values):
    """True once every TODO slot above has been replaced. Otherwise says so."""
    if any(v is ... or v is None for v in values):
        print("A TODO above is still '...'. Fill it in, then run this cell again.")
        return False
    return True

from ep0418 import backtest

if not os.path.exists("spy_daily.csv"):                  # fetched once, then reused
    try:
        import yfinance as yf          # if this fails: run  !pip install yfinance  first
        h = yf.Ticker("SPY").history(period="5y", interval="1d")
        pd.DataFrame({"Date": pd.to_datetime(h.index).tz_localize(None).normalize(),
                      "Close": h["Close"].values}).to_csv("spy_daily.csv", index=False)
    except Exception as e:
        print("Yahoo did not answer:", e, "- wait a minute and run this cell again.")

prices = pd.read_csv("spy_daily.csv", parse_dates=["Date"])
print(f"SPY: {len(prices):,} trading days, {prices.Date.min().date()} to {prices.Date.max().date()}, "
      f"latest ${prices.Close.iloc[-1]:,.2f}")
show.series(prices, col="Close", title="SPY closing price, five years")

position = strategy = None     # placeholders, so later cells give a readable message

## Task 1 — Profile the data, then put buy-and-hold on the board

The prices arrived the third way in the Topic 2B notes: a live API. yfinance
called Yahoo's web service, got JSON back, flattened it to a table, and the setup
cell saved that table as a CSV so you fetch once and reuse. All three forms, in
one cell.

Now the five questions you ask of any dataset you have never seen, in the order
the notes give them. Run the cell and read every line.

In [ ]:
print("1. Rows and columns:", prices.shape, list(prices.columns))
print("2. Period:", prices.Date.min().date(), "to", prices.Date.max().date(),
      "| largest gap between rows:", prices.Date.diff().max().days, "days")
print("3. Close min / max: ${:,.2f} / ${:,.2f}".format(prices.Close.min(), prices.Close.max()))
print("4. Blanks:", int(prices.isna().sum().sum()),
      "| rows with Close <= 0 (a sentinel pretending to be a price):", int((prices.Close <= 0).sum()))
r = prices.Close.pct_change()
print("5. Largest one-day move: {:+.1%} on {}".format(r.loc[r.abs().idxmax()], prices.Date[r.abs().idxmax()].date()))

**What you should see.** About 1,250 rows and two columns, a period of five years,
and a largest gap of three or four days, not one. Ask why before reading on: the
market shuts at weekends and on holidays, so a daily price series has gaps that
are not missing data. Lab 3 makes that distinction precise. No blanks and no zero
prices. The largest one-day move is bigger than most people guess; look its date
up and find out what happened. If any line surprises you, that is the strange
number the pre-lab asked for; write it down.

Before anything clever: what does doing nothing earn?

In [ ]:
bh = backtest.buy_and_hold(prices)
print("Buy-and-hold, five years of SPY")
print(bh)

## Task 2 — A worked example, and what costs do to it

A moving-average crossover: hold SPY when the 20-day average is above the 100-day
average, otherwise cash. Run it twice, with and without costs. Then run it with a
5-day and 20-day pair, which trades far more often, and watch what happens to the
gap between the two scores.

In [ ]:
def crossover(prices, fast=20, slow=100):
    close = prices["Close"]
    f = close.rolling(fast).mean()
    s = close.rolling(slow).mean()
    return (f > s).astype(int).where(s.notna(), 0)     # 0 until the slow window fills

for fast, slow in [(20, 100), (5, 20)]:
    pos = crossover(prices, fast, slow)
    free = backtest.run(prices, pos, cost=0)
    paid = backtest.run(prices, pos)
    print(f"crossover {fast}/{slow}: {paid.trades} trades  "
          f"Sharpe {free.sharpe:.2f} without costs, {paid.sharpe:.2f} with costs  "
          f"(total {free.total:+.1%} -> {paid.total:+.1%})")

**Write down:** which pair trades more, and how much of its return the costs ate.
A rule that trades every few days has to be a lot better than one that trades
every few months, just to pay for itself.

## Task 3 — Read the scorer before you trust it

You are about to spend weeks trying to beat this function. Read it. The cell
prints its source. Then answer, in the cell below, in your own words:

1. Which day's price move does the position you choose on Monday earn?
2. You are long (1) and switch to short (-1). How many units did you trade, and
   what did that cost?
3. Why does the scorer shift your position rather than trusting you to?

In [ ]:
import inspect
print(inspect.getsource(backtest.run))

**My answers:**

1.
2.
3.

## Task 4 — Your strategy

Replace the body of `strategy`. It gets the price table and must return one
position per row. Ask your assistant; then read what it wrote, and run the
checks. `check.no_lookahead` removes the end of the file and reruns your
function. If any earlier decision changes, you were using the future.

> I have a pandas DataFrame `prices` with columns Date and Close, one row per
> trading day. Write a Python function `strategy(prices)` that returns a pandas
> Series of the same length holding 1, 0 or -1 for each day: a trading position
> decided using only that day's row and earlier rows. Use [describe your idea
> here]. Fill the rows before any rolling window is full with 0. Do not use any
> information from later rows. Explain in plain words what it does.

Ideas that are honest to start from: a longer or shorter crossover; a breakout
above the highest close of the last n days; a mean-reversion rule that buys after
a fall of more than x%; a rule that goes to cash when recent volatility is high.

In [ ]:
def strategy(prices):
    # TODO(you): replace this body with your rule. Return 1, 0 or -1 for every row.
    position = ...
    return position


if strategy(prices.head(150)) is not ...:
    position = strategy(prices)

# ---- verification contract — do not edit ----
check.position_is_valid(position, prices)
check.no_lookahead(strategy, prices)
check.trades_at_least(position, minimum=4)
check.report()

In [ ]:
if filled(position):
    mine = backtest.run(prices, position)
    print("Your strategy, five years of SPY"); print(mine)
    print("\nBuy-and-hold"); print(bh)
    check.compare_to_baseline(mine.sharpe, bh.sharpe, metric="Sharpe", lower_is_better=False)
    check.report()
    show.flags(prices.set_index("Date").Close, pd.Series(position).values == 1,
               title="SPY, five years; marked where your rule is long")

### If you got a NOTE

Your strategy did not beat buy-and-hold on the past five years. Most do not; SPY
has spent most of that time going up. That costs you nothing here, and it does not
mean it will lose on the hidden weeks either. Write down what buy-and-hold is
capturing that your rule is not.

## Task 5 — Run your own hidden week first

You cannot see next week. You can imitate it. Pretend the file ends six months
ago: tune your rule on the rows before that date, then score it on the rows
after. The second number is the honest one. Then do it again ending one month
ago, and look at how much a single month can differ from the long run.

If you change your rule after seeing the second number, you have just made the
second number part of the data you tuned on. Do it once, write it down, and
change the rule only if you can say why it should work, not because it did.

In [ ]:
if filled(position):
    cut = prices.Date < prices.Date.max() - pd.Timedelta(days=182)
    before = backtest.run(prices[cut], strategy(prices[cut]))
    after_pos = strategy(prices).loc[~cut]           # decided with the full history, scored on the tail
    after = backtest.run(prices[~cut], after_pos)
    bh_after = backtest.buy_and_hold(prices[~cut])
    print(f"tuned on   {cut.sum():,} days: Sharpe {before.sharpe:.2f}  total {before.total:+.1%}")
    print(f"scored on  {(~cut).sum():,} days: Sharpe {after.sharpe:.2f}  total {after.total:+.1%}"
          f"   (buy-and-hold on the same days: Sharpe {bh_after.sharpe:.2f}, total {bh_after.total:+.1%})")
    month = prices.Date >= prices.Date.max() - pd.Timedelta(days=30)
    last_month = backtest.run(prices[month], strategy(prices).loc[month]); bh_month = backtest.buy_and_hold(prices[month])
    print(f"last month {int(month.sum())} days: return {last_month.total:+.2%}   (buy-and-hold {bh_month.total:+.2%})")

**Did the second number hold up?** If the Sharpe ratio halved or changed sign,
your rule was fitted to the first period rather than to anything real. Say so
below; the write-up prize is for honesty, not for a high number.

## Task 6 — The write-up (half a page, the second podium)

Answer in the cell below. This is what the tutor reads, and it is the part the
module actually grades in CA3.

1. **What the rule assumes about the market.** One or two sentences.
2. **Where it will fail.** The kind of period in which it loses, and why.
3. **Your own hidden-period test.** The two numbers from Task 5, and what you
   concluded.
4. **One mistake the AI assistant made** while writing or changing this rule, how
   you caught it, and what you changed. Everyone has one.

**Write-up**

1.
2.
3.
4.

## Submit

Fill in `ADMIN_NO` and run the cell. It sends your rule, your write-up and your
five-year score to the league, and your entry appears on the scoreboard within a
minute. Run it again whenever you improve the rule; the latest entry counts.
Entries close at the end of Week 7; the exact time is on the scoreboard.

In [ ]:
ADMIN_NO = "..."          # TODO(you): your admin number, e.g. 2512345
WRITEUP = """
TODO(you): paste your four write-up answers here.
"""

FORM_URL = ""
FIELDS = {'admin': '', 'code': '', 'writeup': '', 'sharpe': '', 'total': '', 'trades': '', 'rehearsal': ''}

if ADMIN_NO != "..." and filled(position):
    import inspect, urllib.parse
    try:
        source = inspect.getsource(strategy)
    except OSError:
        source = None
        print("Could not read the function's source. Run the cell that defines "
              "strategy() once more, then run this cell again.")
    mine = backtest.run(prices, position)
    cut = prices.Date < prices.Date.max() - pd.Timedelta(days=182)
    reh_before = backtest.run(prices[cut], strategy(prices[cut])).sharpe
    reh_after = backtest.run(prices[~cut], strategy(prices).loc[~cut]).sharpe
    entry = {"admin": ADMIN_NO, "code": source or "", "writeup": WRITEUP.strip(),
             "sharpe": f"{mine.sharpe:.3f}", "total": f"{mine.total:.4f}", "trades": str(mine.trades),
             "rehearsal": f"before {reh_before:.2f} / after {reh_after:.2f}"}
    if FORM_URL and source:
        data = urllib.parse.urlencode({FIELDS[k]: v for k, v in entry.items()}).encode()
        try:
            urllib.request.urlopen(urllib.request.Request(FORM_URL, data=data,
                                   headers={"User-Agent": "EP0418"}), timeout=60)
            print(f"Entered: {ADMIN_NO}, five-year Sharpe {mine.sharpe:.2f}, {mine.trades} trades. "
                  "Check the scoreboard in a minute.")
        except Exception as e:
            print("The league did not answer:", e, "- try again in a minute.")
    else:
        out = f"strategy_{ADMIN_NO}.py"
        TRIPLE = chr(34) * 3
        with open(out, "w") as f:
            f.write(TRIPLE + WRITEUP.strip() + chr(10) + TRIPLE + chr(10) * 2)
            f.write("import numpy as np" + chr(10) + "import pandas as pd" + chr(10) * 3)
            f.write(source or "# strategy() source missing: see the message above" + chr(10))
        print("saved", out, "- upload it to the League folder on Brightspace.")
        try:
            from google.colab import files
            files.download(out)
        except Exception:
            print("Download it from the folder icon on the left.")

---
## Before you leave, and before the deadline

- The AI mistake from the write-up, below.
- Your Task 5 numbers written down before you saw the scoreboard.
- Come back to this notebook after each lab: Lab 3 gives you `shift()` and moving
  averages on a regular grid, Lab 5 z-scores for a volatility filter, Lab 6 the
  habit of beating a baseline. Replace your entry whenever you have a better rule.

At Lab 7 the tutor shows two tables: the scoreboard on the five years you could
see, and the scoreboard on the four weeks you could not. Look at who moved, and
which way. That reshuffle is the whole lesson of the league.

**Mistake caught:** *(what I expected · what was there · how I found it)*